# GPU verification and tokenizer audit

SemEval 2025 Task 10, Subtask 2: multilabel narrative classification.

This notebook verifies CUDA availability and measures token lengths and truncation for mBERT and XLM-R on the 1,688 eligible training articles. No model weights are loaded and no classifier is trained.

The CUDA check requires a GPU-enabled PyTorch environment. Tokenization runs on CPU. Public tokenizer files can be downloaded from Hugging Face or loaded from the archived audit using `NLP_TOKENIZER_SOURCE`. Dataset texts are processed locally.

The data checks and fixed split protocol match Notebook 01. Tokenizer revisions are pinned to the original experiment. Outputs include tokenizer metadata, per-document lengths, language-level truncation statistics and `tokenizer_audit.zip`.


## 0. Verify CUDA execution

The matrix multiplication verifies a basic CUDA operation. It does not estimate memory requirements or establish the feasibility of model training. The notebook stops if CUDA is unavailable; tokenizer-only verification can be performed separately on CPU.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Runtime CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime and restart the session if required.')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name} | total memory {props.total_memory / 1024**3:.2f} GiB')
with torch.no_grad():
    a = torch.ones((32, 32), device='cuda:0')
    b = a @ a
    torch.cuda.synchronize()
    assert torch.all(b == 32).item()
del a, b
torch.cuda.empty_cache()
print('GPU_OK — CUDA calculation passed; no model loaded.')


## 1. Input and output paths

Defaults target Kaggle. For local execution, set `NLP_INPUT_ROOT` and `NLP_OUTPUT_ROOT` before running this cell. Optional source overrides are `NLP_TRAIN_SOURCE`, `NLP_TEST_SOURCE` and `NLP_KIT_DIR`.

An extracted training/test source override must point to the directory containing `training/` or `test/`, respectively. A kit override must point directly to the directory containing `nlp_setup.py`. If multiple copies are attached, choose explicit paths.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, shutil

INPUT_ROOT = Path(os.environ.get('NLP_INPUT_ROOT', '/kaggle/input'))
OUTPUT_ROOT = Path(os.environ.get('NLP_OUTPUT_ROOT', '/kaggle/working/nlp_setup'))
TRAIN_OVERRIDE = os.environ.get('NLP_TRAIN_SOURCE') or None
TEST_OVERRIDE = os.environ.get('NLP_TEST_SOURCE') or None
KIT_OVERRIDE = os.environ.get('NLP_KIT_DIR') or None
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT_ROOT)
print('Output:', OUTPUT_ROOT)


## 2. Load the support kit and validate the taxonomy

An extracted kit or a ZIP archive is accepted. An archived kit is extracted into the output directory with path-containment checks. A unique support-module directory is required, including when the archive contains a wrapper directory. The selected module is loaded explicitly to avoid reusing a module from an earlier session import.

In [ ]:
if KIT_OVERRIDE:
    KIT_DIR = Path(KIT_OVERRIDE)
else:
    found = sorted({p.parent for p in INPUT_ROOT.rglob('nlp_setup.py')})
    if len(found) == 1:
        KIT_DIR = found[0]
    elif len(found) > 1:
        raise ValueError('Più kit trovati: impostare KIT_OVERRIDE nella cella 1.')
    else:
        archives = list(INPUT_ROOT.rglob('kit_setup_nlp.zip'))
        if len(archives) != 1:
            raise ValueError('Attach exactly one kit_setup_nlp.zip or set KIT_OVERRIDE.')
        KIT_DIR = OUTPUT_ROOT / '_kit'
        KIT_DIR.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as z:
            for info in z.infolist():
                target = (KIT_DIR / info.filename).resolve()
                if not target.is_relative_to(KIT_DIR.resolve()):
                    raise ValueError('Invalid path in kit archive.')
            z.extractall(KIT_DIR)
        extracted = sorted({p.parent for p in KIT_DIR.rglob('nlp_setup.py')})
        if len(extracted) != 1:
            raise ValueError('Extracted archive must contain exactly one support kit.')
        KIT_DIR = extracted[0]

import importlib.util
if not (KIT_DIR / 'nlp_setup.py').is_file():
    raise FileNotFoundError('KIT_DIR must contain nlp_setup.py.')
spec = importlib.util.spec_from_file_location('nlp_setup', KIT_DIR / 'nlp_setup.py')
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)
import numpy as np
import pandas as pd
from IPython.display import display

taxonomy = json.loads((KIT_DIR / 'tassonomia_progetto_v1.json').read_text(encoding='utf-8'))
setup.require(len(taxonomy['coarse']) == 22 and len(taxonomy['fine']) == 96, 'Unexpected taxonomy.')
setup.require(len(set(taxonomy['coarse'])) == 22, 'Duplicate coarse labels.')
setup.require(len(set(taxonomy['fine'])) == 96, 'Duplicate fine labels.')
setup.require(all(l == 'Other' or l.rsplit(': ', 1)[0] in taxonomy['coarse'] for l in taxonomy['fine']),
              'Missing parent label in taxonomy.')
environment = setup.environment_info()
setup.save_json(OUTPUT_ROOT / 'environment.json', environment)
print('Kit:', KIT_DIR)
print('Classes: 22 coarse / 96 fine. Environment:')
display(environment)


## 3. Load training annotations and test texts

Training labels are validated against the project taxonomy. Repeated labels within a document are deduplicated while retaining a fingerprint of the original annotation row. Unknown labels raise an error.

Test annotations are not read. Training records and test texts must match the previously frozen data fingerprints.

In [ ]:
training_source = setup.locate_source(INPUT_ROOT, 'training_NLP.zip', 'training', TRAIN_OVERRIDE)
test_source = setup.locate_source(INPUT_ROOT, 'test_NLP.zip', 'test', TEST_OVERRIDE)
try:
    train_records = setup.load_training(training_source, taxonomy)
    course_texts = setup.load_course_texts(test_source)
finally:
    for source in (training_source, test_source):
        if source.archive is not None:
            source.archive.close()
expected = json.loads((KIT_DIR / 'expected_data_fingerprints.json').read_text(encoding='utf-8'))
setup.require(setup.fingerprint_records(train_records) == expected['training_records'],
              'Training data do not match the verified version.')
setup.require(setup.fingerprint_records(course_texts) == expected['course_texts'],
              'Test texts do not match the verified version.')

data_counts = pd.DataFrame([
    {'language': lang,
     'training': sum(r['language'] == lang for r in train_records),
     'test_texts_only': sum(r['language'] == lang for r in course_texts)}
    for lang in setup.LANGUAGES
])
display(data_counts)
repeated_docs = sum(r['repeated_labels_removed'] > 0 for r in train_records)
print('Training documents with repeated labels deduplicated:', repeated_docs)


## 4. Duplicate and near-duplicate screening

The lexical audit combines whitespace-normalized exact matches with within-language word-trigram TF-IDF cosine similarity of at least 0.80. Candidate links form connected groups. Training documents connected to supplied test texts are excluded from auxiliary training.

The audit threshold is fixed independently of classifier predictions. Audit TF-IDF features are never reused as classifier inputs. This screening cannot rule out translations or all paraphrases, and candidate similarity is not proof of semantic equivalence.

In [ ]:
similarity = setup.audit_similarity(train_records, course_texts, threshold=0.80)
reference = json.loads((KIT_DIR / 'near_duplicate_audit_reference.json').read_text(encoding='utf-8'))
pair_keys = lambda a: {(p['a'], p['b']) for p in a['pairs']}
setup.require(pair_keys(similarity) == pair_keys(reference),
              'Near-duplicate candidates differ from the reference: check inputs and versions.')
setup.require(similarity['excluded_training_keys'] == reference['excluded_training_keys'],
              'Exclusions differ from the frozen reference.')
setup.save_json(OUTPUT_ROOT / 'near_duplicate_audit.json', similarity)
pd.DataFrame(similarity['pairs']).to_csv(OUTPUT_ROOT / 'similarity_pairs.csv', index=False)
print('Candidate pairs:', len(similarity['pairs']))
print('Excluded auxiliary documents:', len(similarity['excluded_training_keys']))
display(pd.DataFrame({'excluded_document': similarity['excluded_training_keys']}))


## 5. Validate the fixed grouped splits

The version-2 manifest assigns the 399 English articles to 395 groups. In particular, `EN_CC_200107.txt` and `EN_CC_200093.txt`, which differ only in quotation marks, share a group.

Each English article appears in exactly one outer-evaluation partition across the three folds. Within a fold, detected groups must not cross training, validation and outer-evaluation partitions. No documents are removed to balance label frequencies.

In [ ]:
manifest = setup.validate_manifest(train_records, similarity, KIT_DIR / 'split_manifest_v2.csv')
split_counts = manifest.groupby(['fold', 'partition']).size().unstack()
display(split_counts[['inner_train', 'inner_validation', 'outer_evaluation']])
manifest.to_csv(OUTPUT_ROOT / 'verified_split_manifest_v2.csv', index=False)
print('OK: 399 English documents; each assigned to outer evaluation exactly once across three folds.')
print('OK: exact-match and candidate English groups do not cross partitions within a fold.')


## 6. Prepare one fold without training

This example builds the first English-only training partition and its multilabel targets. Inner validation is reserved for model and threshold selection; outer predictions are generated only after those choices are fixed. The partitioning code does not technically prevent access to outer labels, so this separation is a methodological requirement.

A fixed seed improves reproducibility but does not guarantee identical results across software versions and hardware.

In [ ]:
FOLD = 1
TRAIN_SEED = 42
MULTILINGUAL = False
setup.seed_everything(TRAIN_SEED)
fold_data = setup.make_fold(train_records, manifest, similarity, FOLD, MULTILINGUAL)
y_train_coarse = setup.binary_labels([r['coarse'] for r in fold_data['inner_train']], taxonomy['coarse'])
y_train_fine = setup.binary_labels([r['fine'] for r in fold_data['inner_train']], taxonomy['fine'])
print('Fold:', FOLD, '| Multilingual:', MULTILINGUAL)
print({role: len(rows) for role, rows in fold_data.items()})
print('Training targets:', y_train_coarse.shape, y_train_fine.shape)
print('Il seed migliora la riproducibilità, ma non garantisce identità tra hardware e versioni diverse.')


## 7. Verify metric compatibility on synthetic cases

The project metric is checked against the original v3 scorer on shared labels and valid synthetic inputs. The comparison covers mean per-document F1 and its document-level standard deviation; the latter is not a confidence interval.

The original scorer ignores labels outside its vocabulary, whereas the strict project implementation rejects unknown labels. Agreement on shared labels does not make the 96-label project taxonomy equivalent to the official taxonomy.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('official_st2_scorer', KIT_DIR / 'official_v3/subtask2_scorer.py')
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
rng = np.random.default_rng(20261103)
for level, filename in [('coarse', 'subtask2_narratives.txt'), ('fine', 'subtask2_subnarratives.txt')]:
    classes = (KIT_DIR / 'official_v3' / filename).read_text(encoding='utf-8').splitlines()
    gold = [[], [classes[0]], [classes[0]], [classes[0], classes[1]]]
    pred = [[], [classes[0]], [], [classes[1]]]
    for _ in range(30):
        gold.append(rng.choice(classes, size=int(rng.integers(0, 5)), replace=False).tolist())
        pred.append(rng.choice(classes, size=int(rng.integers(0, 5)), replace=False).tolist())
    ours = setup.sample_f1_strict(gold, pred, classes)
    theirs = official.evaluate(gold, pred, classes)
    setup.require(np.allclose([ours['mean'], ours['document_std']], theirs, atol=1e-12),
                  'Scorers disagree on synthetic cases.')
print('OK: metric agrees with v3 on synthetic cases using shared labels.')
try:
    setup.sample_f1_strict([['UNKNOWN_LABEL']], [[]], taxonomy['fine'])
except ValueError:
    print('OK: unknown labels are rejected.')
else:
    raise AssertionError('Unknown label was not rejected.')


## 8. Export the setup summary

`setup_summary.json` records counts, fold sizes, environment and limitations. Preserve this file together with the verified manifest and audit outputs. Subsequent notebooks must reconstruct their required variables in their own sessions; saving this notebook does not transfer its in-memory variables.

No classifier training is performed here. The support kit provides `create_run_directory` for storing later experiments in separate directories.

In [ ]:
fold_sizes = {}
for fold in (1, 2, 3):
    fold_sizes[str(fold)] = {}
    for multi in (False, True):
        parts = setup.make_fold(train_records, manifest, similarity, fold, multi)
        fold_sizes[str(fold)]['multilingual' if multi else 'EN_only'] = {k:len(v) for k,v in parts.items()}
summary = {
    'status': 'SETUP_OK', 'split_version': 'v2', 'taxonomy_version': 'progetto_v1_96fine',
    'training_documents_original': len(train_records),
    'course_test_texts': len(course_texts), 'course_test_labels_loaded': False,
    'coarse_classes': len(taxonomy['coarse']), 'fine_classes': len(taxonomy['fine']),
    'similarity_candidate_pairs': len(similarity['pairs']),
    'auxiliary_documents_excluded': len(similarity['excluded_training_keys']),
    'documents_with_repeated_labels': repeated_docs,
    'fold_sizes': fold_sizes, 'environment': environment,
    'limitations': ['Lexical screen does not rule out translations or all paraphrases.',
                    'The supplied test set was consulted during earlier development; it is not an untouched benchmark.',
                    'Extended taxonomy is a project definition, not an official new release.'],
    'training_started': False
}
setup.save_json(OUTPUT_ROOT / 'setup_summary.json', summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Setup complete. Summary:', OUTPUT_ROOT / 'setup_summary.json')


## 9. Load and record the tokenizers

The [mBERT](https://huggingface.co/google-bert/bert-base-multilingual-cased) and [XLM-R](https://huggingface.co/FacebookAI/xlm-roberta-base) revisions below match the archived experiment. Fast tokenizers are required.

By default, files are downloaded from these public repositories. For offline use, set `NLP_TOKENIZER_SOURCE` to the extracted original `tokenizer_audit` directory containing `tokenizer_metadata.json`, `mBERT/` and `XLM-R/`. The recorded revisions and file hashes are verified before loading. The source archive and metadata must be retained together.

Saved file hashes identify the serialization produced by the current environment; they may differ between library versions even when the source tokenizer is unchanged.

In [ ]:
from transformers import AutoTokenizer
import hashlib

TOKEN_DIR = OUTPUT_ROOT / 'tokenizer_audit'
TOKEN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_IDS = {
    'mBERT': 'google-bert/bert-base-multilingual-cased',
    'XLM-R': 'FacebookAI/xlm-roberta-base',
}
MODEL_REVISIONS = {
    'mBERT': '3f076fdb1ab68d5b2880cb87a0886f315b8146f8',
    'XLM-R': 'e73636d4f797dec63c3081bb6ed5c7b0bb3f2089',
}
source_path = os.environ.get('NLP_TOKENIZER_SOURCE')
tokenizer_source = Path(source_path) if source_path else None
source_metadata = None
if tokenizer_source is not None:
    source_metadata = json.loads(
        (tokenizer_source / 'tokenizer_metadata.json').read_text(encoding='utf-8')
    )

tokenizers = {}
tokenizer_metadata = {}
for name, model_id in MODEL_IDS.items():
    print('Loading tokenizer:', name, flush=True)
    revision = MODEL_REVISIONS[name]
    if tokenizer_source is not None:
        saved = source_metadata[name]
        setup.require(saved['model_id'] == model_id and saved['revision'] == revision,
                      'Archived tokenizer identity does not match the pinned revision.')
        source_dir = (tokenizer_source / name).resolve()
        setup.require(bool(saved['files_sha256']), 'Archived tokenizer hashes are missing.')
        for filename, expected_hash in saved['files_sha256'].items():
            file_path = (source_dir / filename).resolve()
            setup.require(file_path.is_relative_to(source_dir), 'Invalid tokenizer file path.')
            setup.require(hashlib.sha256(file_path.read_bytes()).hexdigest() == expected_hash,
                          f'Tokenizer file hash mismatch: {name}/{filename}')
        tok = AutoTokenizer.from_pretrained(
            source_dir, local_files_only=True, use_fast=True, trust_remote_code=False
        )
    else:
        tok = AutoTokenizer.from_pretrained(
            model_id, revision=revision, use_fast=True, trust_remote_code=False
        )
    setup.require(tok.is_fast, 'A fast tokenizer is required.')
    dest = TOKEN_DIR / name
    tok.save_pretrained(dest)
    tokenizer_metadata[name] = {
        'model_id': model_id, 'revision': revision,
        'tokenizer_class': type(tok).__name__, 'is_fast': tok.is_fast,
        'model_max_length': int(tok.model_max_length),
        'special_tokens_single_sequence': int(tok.num_special_tokens_to_add(pair=False)),
        'files_sha256': {
            p.name: hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(dest.iterdir()) if p.is_file()
        },
    }
    tokenizers[name] = tok
setup.save_json(TOKEN_DIR / 'tokenizer_metadata.json', tokenizer_metadata)
print(json.dumps(tokenizer_metadata, ensure_ascii=False, indent=2))
print('TOKENIZERS_OK — tokenizers loaded; no model weights loaded.')


## 10. Measure token lengths and truncation

Content tokens are counted without truncation for all eligible training articles. For each total length limit (128, 256, 384 and 512), the content capacity excludes the tokenizer's special tokens.

`mean_content_tokens_lost_pct` averages the lost-token fraction across documents; `pooled_content_tokens_lost_pct` divides all lost tokens by all content tokens within a language. These measures describe token removal, not the amount of task-relevant information lost. They do not establish which segment-selection strategy performs best.

All languages are audited separately. These descriptive measurements use no test annotations or model performance scores.

In [ ]:
active_records = sorted([r for r in train_records if r['key'] not in set(similarity['excluded_training_keys'])],
                        key=lambda r:r['key'])
setup.require(len(active_records)==1688,'Unexpected eligible training count.')
length_rows = []; truncation_rows = []
for name, tok in tokenizers.items():
    print('Measuring token lengths:', name, flush=True)
    lengths = []
    for start in range(0,len(active_records),32):
        batch=active_records[start:start+32]
        encoded=tok([r['text'] for r in batch],add_special_tokens=False,
                    truncation=False,padding=False,return_attention_mask=False,
                    return_token_type_ids=False,verbose=False)
        lengths.extend(len(ids) for ids in encoded['input_ids'])
    setup.require(len(lengths) == len(active_records), 'Token counts and document counts differ.')
    special=tok.num_special_tokens_to_add(pair=False)
    for record,n in zip(active_records,lengths):
        length_rows.append({'model':name,'language':record['language'],
            'document_id':record['document_id'],'content_tokens':n,
            'total_tokens_with_special':n+special})
    for language in setup.LANGUAGES:
        n=np.array([length for r,length in zip(active_records,lengths) if r['language']==language])
        for limit in (128,256,384,512):
            setup.require(limit<=tok.model_max_length,'Length limit exceeds tokenizer capacity.')
            capacity=limit-special
            lost=np.maximum(n-capacity,0)
            fractions=np.divide(lost,n,out=np.zeros(len(n),dtype=float),where=n>0)
            truncation_rows.append({'model':name,'language':language,'documents':len(n),
                'max_length':limit,'content_capacity':capacity,
                'median_total_tokens':float(np.median(n+special)),
                'p90_total_tokens':float(np.quantile(n+special,.9)),
                'truncated_documents_pct':float(100*np.mean(lost>0)),
                'mean_content_tokens_lost_pct':float(100*fractions.mean()),
                'pooled_content_tokens_lost_pct':float(100*lost.sum()/n.sum())})
length_table=pd.DataFrame(length_rows)
truncation_table=pd.DataFrame(truncation_rows)
length_table.to_csv(TOKEN_DIR/'document_token_lengths.csv',index=False)
truncation_table.to_csv(TOKEN_DIR/'truncation_by_language.csv',index=False)
display(truncation_table.loc[truncation_table.language=='EN',
    ['model','max_length','documents','median_total_tokens','p90_total_tokens',
     'truncated_documents_pct','mean_content_tokens_lost_pct']])
print('TOKEN_LENGTHS_OK — counts saved for both tokenizers and all languages.')


## 11. Export the audit

`tokenizer_summary.json` records the environment, pinned revisions and English truncation statistics. The ZIP archive also contains saved tokenizers, metadata and full per-language tables.

The CUDA check establishes basic device availability only. Training memory must be assessed separately using the intended model and optimization configuration.

In [ ]:
tokenizer_summary={'status':'TOKENIZER_AUDIT_OK','training_documents':len(active_records),
    'models':tokenizer_metadata,'environment':environment,
    'english_truncation':truncation_table[truncation_table.language=='EN'].to_dict('records'),
    'course_test_labels_loaded':False,'model_weights_loaded':False,'training_started':False}
setup.save_json(TOKEN_DIR/'tokenizer_summary.json',tokenizer_summary)
archive=shutil.make_archive(str(TOKEN_DIR),'zip',TOKEN_DIR)
print(json.dumps(tokenizer_summary,ensure_ascii=False,indent=2))
print('Audit archive:',archive)
